In [1]:
import pandas as pd
import re
from pathlib import Path

# -----------------------------
# 1. Load data
# -----------------------------

INPUT_PATH = Path("post_table_translation_preprocessed.csv")
OUTPUT_PATH = Path("annotation_sample_3000_with_batches.csv")

RANDOM_STATE = 42

df = pd.read_csv(INPUT_PATH)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

TEXT_COL = "translated_clean"

if TEXT_COL not in df.columns:
    raise ValueError(f"Column '{TEXT_COL}' not found. Available columns: {df.columns.tolist()}")

df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)

# Keep only rows with usable translated text
df = df[df[TEXT_COL].str.strip().ne("")].copy()

# Avoid duplicate posts if post_id exists
if "post_id" in df.columns:
    df = df.drop_duplicates(subset=["post_id"]).copy()

df = df.reset_index(drop=True)
df["source_row_number"] = df.index

print("Rows after cleaning:", len(df))

Rows: 20000
Columns: ['post_id', 'utility_id', 'event_id', 'created_at', 'language', 'native_text', 'translated_text', 'reaction_count', 'comment_count', 'share_count', 'media_type', 'url', 'native_text_hashtags', 'translated_text_hashtags', 'year', 'month', 'day', 'native_text_no_hashtags', 'translated_text_no_hashtags', 'native_clean', 'translated_clean']
Rows after cleaning: 19978


In [2]:
# -----------------------------
# 2. Define keyword groups
# -----------------------------

keyword_groups = {
    "water_quality_safety_contamination": [
        "water quality", "drinking water", "tap water", "safe to drink",
        "water safety", "contamination", "contaminated", "pollution",
        "turbidity", "chlorine", "bacteria", "e coli", "ecoli",
        "coliform", "legionella", "pfas", "boil water", "boil",
        "water sampling", "water monitoring", "quality monitoring",
        "public health", "health risk", "beach quality", "water test",
        "water testing", "water analysis"
    ],

    "service_disruption_repair_infrastructure": [
        "water interruption", "service interruption", "water service interruption",
        "interruption", "disruption", "outage", "no water", "water cut",
        "low pressure", "pressure", "pipe burst", "burst pipe", "broken pipe",
        "leak", "fault", "repair", "maintenance", "construction",
        "water network", "network", "restoration time", "water supply interruption",
        "service alert", "planned maintenance", "emergency repair"
    ],

    "water_conservation_drought_demand": [
        "drought", "scarcity", "water scarcity", "water restriction",
        "water restrictions", "water conservation", "water saving",
        "save water", "conserve water", "water consumption", "water demand",
        "heatwave", "heat wave", "dry spell", "sustainable water use",
        "conservation tips", "watering ban", "water wisely"
    ],

    "flood_stormwater_wastewater_sewer": [
        "flood", "flooding", "storm", "stormwater", "heavy rain",
        "rainfall", "rainwater", "runoff", "drainage", "drain",
        "sewer", "sewerage", "wastewater", "wastewater treatment",
        "overflow", "combined sewer overflow", "sewage", "treatment plant"
    ],

    "environment_waste_recycling_biodiversity": [
        "waste sorting", "waste management", "waste collection", "recycling",
        "hazardous waste", "plastic waste", "food waste", "biowaste",
        "bio waste", "battery", "circular economy", "biodiversity",
        "wildlife", "ecosystem", "baltic sea", "air quality",
        "environmental conservation", "sustainability", "climate change",
        "invasive species", "compost", "organic waste"
    ],

    "education_awareness_campaign": [
        "education", "educational", "awareness", "campaign", "outreach",
        "school", "children", "student", "students", "workshop", "training",
        "tour", "museum", "exhibition", "world water day", "tips",
        "public awareness", "learn", "learning", "webinar"
    ],

    "routine_institutional_community": [
        "holiday", "christmas", "greeting", "celebration", "anniversary",
        "award", "event", "community engagement", "community events",
        "partnership", "customer service", "billing", "bill", "payment",
        "job", "recruitment", "career", "contest", "cultural", "heritage",
        "promotion", "brand", "opening hours", "office closed"
    ],
}

In [3]:
# -----------------------------
# 3. Assign sampling groups
# -----------------------------

priority_order = [
    "water_quality_safety_contamination",
    "service_disruption_repair_infrastructure",
    "flood_stormwater_wastewater_sewer",
    "water_conservation_drought_demand",
    "environment_waste_recycling_biodiversity",
    "education_awareness_campaign",
    "routine_institutional_community",
]

compiled_patterns = {
    group: re.compile("|".join(re.escape(term) for term in terms), re.IGNORECASE)
    for group, terms in keyword_groups.items()
}

def assign_sampling_group(text):
    for group in priority_order:
        if compiled_patterns[group].search(text):
            return group
    return "other_or_unclear"

def get_matched_keywords(text, group):
    if group not in keyword_groups:
        return ""

    matched = [
        term for term in keyword_groups[group]
        if term.lower() in text.lower()
    ]

    return "; ".join(sorted(set(matched)))

df["sampling_group"] = df[TEXT_COL].apply(assign_sampling_group)

df["matched_keywords"] = df.apply(
    lambda row: get_matched_keywords(row[TEXT_COL], row["sampling_group"]),
    axis=1
)

print(df["sampling_group"].value_counts())

sampling_group
other_or_unclear                            5405
water_quality_safety_contamination          3636
education_awareness_campaign                2725
service_disruption_repair_infrastructure    2578
environment_waste_recycling_biodiversity    1927
flood_stormwater_wastewater_sewer           1857
routine_institutional_community             1494
water_conservation_drought_demand            356
Name: count, dtype: int64


In [4]:
# -----------------------------
# 4. Add crisis-priority signals
# -----------------------------

rare_terms = re.compile(
    r"boil|contaminat|bacteria|e coli|ecoli|legionella|pfas|"
    r"outbreak|illness|turbidity|overflow|flood|drought|restriction|"
    r"outage|interruption|pipe burst|burst pipe|broken pipe|"
    r"low pressure|no water|sewer|wastewater",
    re.IGNORECASE
)

df["rare_or_crisis_signal"] = df[TEXT_COL].str.contains(rare_terms, na=False)

print(df["rare_or_crisis_signal"].value_counts())

rare_or_crisis_signal
False    16042
True      3936
Name: count, dtype: int64


In [5]:
# -----------------------------
# 5. Define annotation batches
# -----------------------------

batch_targets = {
    "Batch 1": {
        "water_quality_safety_contamination": 300,
        "service_disruption_repair_infrastructure": 250,
        "flood_stormwater_wastewater_sewer": 200,
        "water_conservation_drought_demand": 180,
        "environment_waste_recycling_biodiversity": 160,
        "education_awareness_campaign": 160,
        "routine_institutional_community": 180,
        "other_or_unclear": 70,
    },

    "Batch 2": {
        "water_quality_safety_contamination": 100,
        "service_disruption_repair_infrastructure": 90,
        "flood_stormwater_wastewater_sewer": 60,
        "water_conservation_drought_demand": 70,
        "environment_waste_recycling_biodiversity": 50,
        "education_awareness_campaign": 50,
        "routine_institutional_community": 60,
        "other_or_unclear": 20,
    },

    "Batch 3": {
        "water_quality_safety_contamination": 200,
        "service_disruption_repair_infrastructure": 160,
        "flood_stormwater_wastewater_sewer": 90,
        "water_conservation_drought_demand": 100,
        "environment_waste_recycling_biodiversity": 90,
        "education_awareness_campaign": 90,
        "routine_institutional_community": 160,
        "other_or_unclear": 110,
    },
}

In [6]:
# -----------------------------
# 6. Generate staged annotation sample
# -----------------------------

selected_indices = set()
batch_samples = []

for batch_name, targets in batch_targets.items():

    group_samples = []

    for group, n in targets.items():

        group_df = df[
            (df["sampling_group"] == group)
            & (~df.index.isin(selected_indices))
        ].copy()

        if group_df.empty:
            print(f"Warning: no rows available for {group} in {batch_name}")
            continue

        # Put stronger crisis/risk examples first
        group_df = group_df.sort_values(
            ["rare_or_crisis_signal"],
            ascending=[False]
        )

        take_n = min(n, len(group_df))

        # Sample from the prioritized pool to avoid only taking near-duplicates
        candidate_pool = group_df.head(
            max(take_n, int(len(group_df) * 0.75))
        )

        sample = candidate_pool.sample(
            n=take_n,
            random_state=RANDOM_STATE
        )

        sample["annotation_batch"] = batch_name

        group_samples.append(sample)
        selected_indices.update(sample.index.tolist())

    batch_df = pd.concat(group_samples, ignore_index=False)

    batch_df = batch_df.sample(
        frac=1,
        random_state=RANDOM_STATE
    ).reset_index(drop=True)

    batch_samples.append(batch_df)

annotation_sample = pd.concat(batch_samples, ignore_index=True)

annotation_sample.insert(
    0,
    "annotation_id",
    range(1, len(annotation_sample) + 1)
)

print("Total annotation sample:", len(annotation_sample))
print(annotation_sample["annotation_batch"].value_counts())

Total annotation sample: 3000
annotation_batch
Batch 1    1500
Batch 3    1000
Batch 2     500
Name: count, dtype: int64


In [7]:
# -----------------------------
# 7. Add manual annotation columns
# -----------------------------

annotation_sample["task_a_gold"] = ""
annotation_sample["task_b_gold"] = ""
annotation_sample["annotator_notes"] = ""

# Suggested Task A labels:
# - Water quality / safety / public health
# - Service disruption / repair / infrastructure
# - Water conservation / drought / demand management
# - Flood / stormwater / wastewater / sewer
# - Environmental sustainability / waste / recycling / biodiversity
# - Public education / heritage / community engagement
# - Routine / institutional / customer service / other

# Suggested Task B labels:
# - Alert / warning
# - Instruction / advice to public
# - Operational update / resolution
# - Reassurance / safety information
# - Education / awareness
# - Institutional promotion / community news

In [8]:
# -----------------------------
# 8. Select output columns
# -----------------------------

base_cols = [
    "annotation_id",
    "annotation_batch",
    "source_row_number",
]

if "post_id" in annotation_sample.columns:
    base_cols.append("post_id")

metadata_cols = [
    col for col in [
        "utility_id",
        "created_at",
        "year",
        "month",
        "language",
        "reaction_count",
        "comment_count",
        "share_count",
        "media_type",
        "url",
    ]
    if col in annotation_sample.columns
]

text_cols = [
    col for col in [
        "native_text",
        "translated_text",
        "native_clean",
        "translated_clean",
        "native_text_hashtags",
        "translated_text_hashtags",
    ]
    if col in annotation_sample.columns
]

sampling_cols = [
    "sampling_group",
    "matched_keywords",
    "rare_or_crisis_signal",
    "task_a_gold",
    "task_b_gold",
    "annotator_notes",
]

keep_cols = base_cols + metadata_cols + text_cols + sampling_cols

annotation_sample = annotation_sample[keep_cols]

annotation_sample.head()

,annotation_id,annotation_batch,source_row_number,post_id,utility_id,created_at,year,month,language,reaction_count,...,native_clean,translated_clean,native_text_hashtags,translated_text_hashtags,sampling_group,matched_keywords,rare_or_crisis_signal,task_a_gold,task_b_gold,annotator_notes
0,1,Batch 1,14101,482407545146552,U04,2013-03-15 13:25:33+00:00,2013,3,Norwegian,20.0,...,gratulerer oslo vi er norges beste klimaby nor...,congratulations oslo we are norway s best clim...,NaN,NaN,education_awareness_campaign,campaign,False,,,
1,2,Batch 1,18219,1338919832798326,U03,2016-12-05 15:28:05+00:00,2016,12,Portuguese,18.0,...,epal alarga horário de atendimento na loja da ...,epal extends service hours at the av da liberd...,#EPAL,#EPAL,routine_institutional_community,customer service,False,,,
2,3,Batch 1,8666,1293598662785386,U02,2025-10-16 11:01:23+00:00,2025,10,German,151.0,...,eis eis baby aber diesmal für unser sielnetz s...,ice ice baby but this time for our sewer netwo...,#bauenfürhamburg #zusammenfürwasser,#buildingforhamburg #togetherforwater,service_disruption_repair_infrastructure,network,True,,,
3,4,Batch 1,16484,1282777398465618,U01,2017-03-11 19:49:22+00:00,2017,3,Finnish,1.0,...,huomio helsingin hietalahdenranta putkirikon t...,attention helsinki hietalahti due to a pipe bu...,NaN,NaN,service_disruption_repair_infrastructure,pipe burst; repair,True,,,
4,5,Batch 1,13625,5304355719588031,U03,2022-04-07 14:28:22+00:00,2022,4,Portuguese,30.0,...,presidente da epal participa na grande conferê...,epal s president participates in the great con...,NaN,NaN,service_disruption_repair_infrastructure,network,False,,,


In [9]:
# -----------------------------
# 9. Save annotation file
# -----------------------------

annotation_sample.to_csv(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)
print("Total rows:", len(annotation_sample))

print("\nBatch counts:")
print(annotation_sample["annotation_batch"].value_counts())

print("\nSampling group by batch:")
print(
    pd.crosstab(
        annotation_sample["sampling_group"],
        annotation_sample["annotation_batch"]
    )
)

Saved: annotation_sample_3000_with_batches.csv
Total rows: 3000

Batch counts:
annotation_batch
Batch 1    1500
Batch 3    1000
Batch 2     500
Name: count, dtype: int64

Sampling group by batch:
annotation_batch                          Batch 1  Batch 2  Batch 3
sampling_group                                                     
education_awareness_campaign                  160       50       90
environment_waste_recycling_biodiversity      160       50       90
flood_stormwater_wastewater_sewer             200       60       90
other_or_unclear                               70       20      110
routine_institutional_community               180       60      160
service_disruption_repair_infrastructure      250       90      160
water_conservation_drought_demand             180       70      100
water_quality_safety_contamination            300      100      200


In [10]:
# -----------------------------
# 10. Save one CSV file per annotation batch
# -----------------------------

output_dir = Path("annotation_batches")
output_dir.mkdir(parents=True, exist_ok=True)

for batch_name, batch_df in annotation_sample.groupby("annotation_batch"):
    batch_number = batch_name.lower().replace(" ", "_")

    batch_path = output_dir / f"annotation_sample_{batch_number}.csv"

    batch_df.to_csv(batch_path, index=False)

    print(f"Saved {batch_name}: {len(batch_df)} rows -> {batch_path}")

Saved Batch 1: 1500 rows -> annotation_batches\annotation_sample_batch_1.csv
Saved Batch 2: 500 rows -> annotation_batches\annotation_sample_batch_2.csv
Saved Batch 3: 1000 rows -> annotation_batches\annotation_sample_batch_3.csv


In [11]:
# -----------------------------
# 11. Create 200-row repeat annotation sample for self-consistency / Cohen's kappa
# -----------------------------

REPEAT_SAMPLE_SIZE = 200
REPEAT_OUTPUT_PATH = Path("repeat_annotation_200_for_kappa.csv")

# Take the repeat sample from Batch 1 first, because it is the minimum paper-ready set.
# Stratify by sampling_group so all main categories are represented.

batch1_df = annotation_sample[
    annotation_sample["annotation_batch"] == "Batch 1"
].copy()

repeat_parts = []

# Proportional stratified sample by sampling_group
group_counts = batch1_df["sampling_group"].value_counts(normalize=True)

for group, prop in group_counts.items():
    group_df = batch1_df[batch1_df["sampling_group"] == group].copy()

    n = round(prop * REPEAT_SAMPLE_SIZE)
    n = min(n, len(group_df))

    sampled = group_df.sample(
        n=n,
        random_state=RANDOM_STATE
    )

    repeat_parts.append(sampled)

repeat_sample = pd.concat(repeat_parts, ignore_index=True)

# Fix rounding so the final sample is exactly 200 rows
if len(repeat_sample) > REPEAT_SAMPLE_SIZE:
    repeat_sample = repeat_sample.sample(
        n=REPEAT_SAMPLE_SIZE,
        random_state=RANDOM_STATE
    )

elif len(repeat_sample) < REPEAT_SAMPLE_SIZE:
    missing_n = REPEAT_SAMPLE_SIZE - len(repeat_sample)

    remaining_pool = batch1_df[
        ~batch1_df["annotation_id"].isin(repeat_sample["annotation_id"])
    ]

    extra = remaining_pool.sample(
        n=missing_n,
        random_state=RANDOM_STATE
    )

    repeat_sample = pd.concat([repeat_sample, extra], ignore_index=True)

repeat_sample = repeat_sample.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

repeat_sample.insert(
    0,
    "repeat_annotation_id",
    range(1, len(repeat_sample) + 1)
)

# Create separate blank columns for the second annotation round
repeat_sample["task_a_gold_round_2"] = ""
repeat_sample["task_b_gold_round_2"] = ""
repeat_sample["round_2_notes"] = ""

repeat_sample.to_csv(REPEAT_OUTPUT_PATH, index=False)

print("Saved:", REPEAT_OUTPUT_PATH)
print("Rows:", len(repeat_sample))

print("\nRepeat sample distribution:")
print(repeat_sample["sampling_group"].value_counts())

print("\nPercentage distribution:")
print(
    (repeat_sample["sampling_group"].value_counts(normalize=True) * 100)
    .round(1)
)

Saved: repeat_annotation_200_for_kappa.csv
Rows: 200

Repeat sample distribution:
sampling_group
water_quality_safety_contamination          40
service_disruption_repair_infrastructure    34
flood_stormwater_wastewater_sewer           27
water_conservation_drought_demand           24
routine_institutional_community             24
education_awareness_campaign                21
environment_waste_recycling_biodiversity    21
other_or_unclear                             9
Name: count, dtype: int64

Percentage distribution:
sampling_group
water_quality_safety_contamination          20.0
service_disruption_repair_infrastructure    17.0
flood_stormwater_wastewater_sewer           13.5
water_conservation_drought_demand           12.0
routine_institutional_community             12.0
education_awareness_campaign                10.5
environment_waste_recycling_biodiversity    10.5
other_or_unclear                             4.5
Name: proportion, dtype: float64
